# TotalSegmentator inference (nb2): converted NIfTI → segmentations  —  model-specific

Runs TotalSegmentator (v1.5.6, `--ml` multilabel) on the GPU VM. Consumes the
**Boundary-A** archive `converted_nifti.tar.lz4` from nb1 and emits the **Boundary-B**
archive `segmentations.tar.lz4` with the canonical layout:
```
<SeriesInstanceUID>/<model>/segmentations/<SeriesInstanceUID>.nii.gz
<SeriesInstanceUID>/<model>/label_map.json      # {label_id: label_name}
```
TotalSegmentator's `--ml` already produces one multilabel volume; the authoritative
`{label_id: label_name}` comes from `totalsegmentator.map_to_binary.class_map['total']`,
so no label IDs are ever hand-transcribed. nb3 (shared) turns this into DICOM-SEG +
pyradiomics + SR using the `total` rows of the SNOMED mapping.

## Imports

In [ ]:
import json
import shutil
import subprocess
import time
import traceback
from pathlib import Path

NOTEBOOK_START = time.time()
def _elapsed(s=None):
    return f"{time.time() - (s if s is not None else NOTEBOOK_START):.1f}s"
print(f"[T+{_elapsed()}] Imports complete")

## Parameters

In [ ]:
# Boundary-A archive produced by nb1 (local file on the same VM).
converted_nifti_path = "converted_nifti.tar.lz4"

# Short model identifier used in the Boundary-B layout (<uid>/<model>/...).
model_name = "total"

# 'cuda' for GPU, 'cpu' for CPU-only.
accelerator = "cuda"

# Reserved for checkpoint/resume on preemption (not yet wired in this notebook).
checkpoint_gcs = ""

# Model-specific knob injected via `papermill -f inference_params.yaml`.
# fast=True uses TotalSegmentator's 3mm model (faster, lower resolution).
fast = False

## Extract Boundary-A archive

In [ ]:
NIFTI_DIR = Path('/tmp/converted_nifti')
SEG_DIR = Path('/tmp/segmentations')
for _d in (NIFTI_DIR, SEG_DIR):
    if _d.exists():
        shutil.rmtree(_d)
    _d.mkdir(parents=True, exist_ok=True)

subprocess.run(f'lz4 -d -c {converted_nifti_path} | tar -xf - -C {NIFTI_DIR.parent}',
               shell=True, check=True)
if (NIFTI_DIR / 'converted_nifti').is_dir():
    NIFTI_DIR = NIFTI_DIR / 'converted_nifti'
series_uids = sorted(d.name for d in NIFTI_DIR.iterdir() if d.is_dir())
print(f'Series : {len(series_uids)}  |  fast={fast}')

## Authoritative label map from TotalSegmentator's class map

In [ ]:
from totalsegmentator.map_to_binary import class_map
# {label_id(int): label_name(str)} for the 104-structure 'total' task.
TOTAL_LABELS = {str(k): v for k, v in class_map['total'].items()}
print(f'Loaded {len(TOTAL_LABELS)} TotalSegmentator label ids')

## Run TotalSegmentator → Boundary-B layout

In [ ]:
errors = []
usage_metrics = {'series': {}}

for uid in series_uids:
    nii = NIFTI_DIR / uid / f'{uid}.nii.gz'
    if not nii.exists():
        cands = list((NIFTI_DIR / uid).glob('*.nii.gz'))
        if not cands:
            errors.append(f'{uid}: no NIfTI found')
            continue
        nii = cands[0]
    work = Path('/tmp/ts_work') / uid
    if work.exists():
        shutil.rmtree(work)
    work.mkdir(parents=True, exist_ok=True)
    # TS 1.5.6 treats -o as an output *name*: with --ml it writes either
    # <out>.nii (a file) or <out>/segmentations.nii (a dir). Use the proven
    # extension-less name and locate the produced volume by rglob so both
    # layouts work (matches inferenceTotalSegmentatorNotebook.ipynb).
    out_dir = work / 'segmentations'
    cmd = ['TotalSegmentator', '-i', str(nii), '-o', str(out_dir), '--ml']
    if fast:
        cmd.append('--fast')
    print(f'[T+{_elapsed()}] {uid}: {" ".join(cmd)}', flush=True)
    t0 = time.time()
    try:
        res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        if res.returncode != 0:
            errors.append(f'{uid}: TotalSegmentator rc={res.returncode}\n{res.stderr}')
            print(f'  ERROR rc={res.returncode}: {res.stderr[:500]}')
            continue
        # Locate the multilabel volume whether TS wrote a file or a directory.
        produced = next(iter(sorted(work.rglob('*.nii'))), None)
        if produced is None:
            errors.append(f'{uid}: no multilabel NIfTI produced')
            continue
        dest = SEG_DIR / uid / model_name / 'segmentations'
        dest.mkdir(parents=True, exist_ok=True)
        # Compress .nii -> .nii.gz into the canonical name.
        subprocess.run(f'gzip -c "{produced}" > "{dest / (uid + ".nii.gz")}"',
                       shell=True, check=True)
        (SEG_DIR / uid / model_name / 'label_map.json').write_text(
            json.dumps({'model': model_name, 'labels': TOTAL_LABELS}, indent=2))
        # Propagate the exact input NIfTI (identical geometry to the mask) to the
        # Boundary-B series root so nb3 runs radiomics against it directly instead
        # of a second, independent dcm2niix conversion of the reference DICOM.
        shutil.copy(str(nii), str(SEG_DIR / uid / 'reference.nii.gz'))
        usage_metrics['series'][uid] = {'model_inference_s': round(time.time() - t0, 1)}
        print(f'  done in {usage_metrics["series"][uid]["model_inference_s"]}s')
    except Exception as exc:
        errors.append(f'{uid}: {traceback.format_exc()}')
        print(f'  ERROR: {exc}')
    finally:
        shutil.rmtree(work, ignore_errors=True)

if errors:
    Path('inference_errors.txt').write_text('\n'.join(errors))
print(f'[T+{_elapsed()}] Inference complete ({len(errors)} error(s))')

## Package Boundary-B archive + usage metrics

In [ ]:
import csv
produced = [d for d in SEG_DIR.iterdir() if d.is_dir()]
if not produced:
    raise RuntimeError('No segmentations produced — see inference_errors.txt')

subprocess.run(f'tar -cf - -C {SEG_DIR.parent} {SEG_DIR.name} | lz4 > segmentations.tar.lz4',
               shell=True, check=True)
size_mb = Path('segmentations.tar.lz4').stat().st_size / (1024 ** 2)

usage_metrics['total_elapsed_s'] = round(time.time() - NOTEBOOK_START, 1)
with open('inference_UsageMetrics.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['SeriesInstanceUID', 'model', 'model_inference_s', 'run_total_elapsed_s'])
    for uid, m in usage_metrics['series'].items():
        w.writerow([uid, model_name, m.get('model_inference_s', ''), usage_metrics['total_elapsed_s']])

print(f'[T+{_elapsed()}] Wrote segmentations.tar.lz4 ({size_mb:.1f} MB, {len(produced)} series)')